# Introduction to JAX+Flax


Why should one learn JAX if there are already so many other deep learning frameworks like PyTorch and TensorFlow? 

**The short answer: because it can be extremely fast.**

For instance, a small GoogleNet on CIFAR10 can be trained in JAX 3x faster than in PyTorch with a similar setup. Note that for larger models, larger batch sizes, or smaller GPUs, a considerably smaller speedup is expected, and the code has not been designed for benchmarking. 

Nonetheless, JAX enables this speedup by** compiling functions and numerical programs for accelerators (GPU/TPU) _just in time_**, finding the optimal utilization of the hardware.

Frameworks with dynamic computation graphs like **PyTorch cannot achieve the same efficiency**, since they **cannot anticipate the next operations before the user calls them**. For example, in an Inception block of GoogleNet, we apply multiple convolutional layers in parallel on the same input. _JAX_ can **optimize the execution of this layer by compiling the whole forward pass for the available accelerator and fusing operations where possible, reducing memory access and speeding up execution**. In contrast, when calling the first convolutional layer in _PyTorch_, the framework does not know that multiple convolutions on the same feature map will follow. It **sends each operation one by one to the GPU, and can only adapt the execution after seeing the next Python calls**. Hence, JAX can make more efficient use of the GPU than, for instance, PyTorch.

However, everything comes with a price. In order to efficiently compile programs just-in-time in JAX, **the functions need to be written with certain constraints**:
* Firstly, the functions are **not allowed to have side-effects**, meaning that they are **not allowed to affect any variable outside of their namespaces**. For instance, in-place operations affect a variable even outside of the function. 
* Moreover, **stochastic operations** such as `torch.rand(...)` change the global state of pseudo random number generators, which **is not allowed in functional JAX**. 
* Secondly, JAX **compiles the functions based on the anticipated shapes of all arrays/tensors in the function**. This becomes **problematic if the shapes or the program flow within the function depend on the values of the tensor**. For instance, in the operation `y = x[x>3]`, the shape of y depends on how many values of x are greater than 3. 
* Still, in most common cases of training neural networks, it is straightforward to write functions within these constraints.


In [1]:
## Standard libraries
import os
import math
import numpy as np
import time

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib.colors import to_rgba

# Checkout: [Deprecated use of IPython.display.set_matplotlib_formats #1842](https://github.com/scverse/scanpy/issues/1842#issuecomment-2713798008)
import IPython.display
IPython.display.set_matplotlib_formats = set_matplotlib_formats

set_matplotlib_formats('svg', 'pdf') # For export

import seaborn as sns
sns.set()

## Progress bar
from tqdm.notebook import tqdm

## JAX as NumPy on accelerators

Every deep learning framework has its own API for dealing with data arrays. 

For example, PyTorch uses `torch.Tensor` as data arrays on which it defines several operations like matrix multiplication, taking the mean of the elements, etc. 

In **JAX**, this basic **API strongly resembles that of NumPy**, and even has the same name in JAX (`jax.numpy`). So, for now, let’s think of JAX as NumPy that runs on accelerators. 

Importing the `jax` and `jax.numpy`.

The NumPy API of JAX is usually imported as `jnp`, to keep a resemblance to NumPy’s import as `np`.

In [3]:
import jax
import jax.numpy as jnp

print("Using jax", jax.__version__)

Using jax 0.8.2


### Device Arrays

We can use `jnp.zeros()` and `jnp.arange()` to create arrays just like numpy and pytorch.

In [ ]:
a = jnp.zeros((2, 5), dtype=jnp.float32)
b = jnp.arange(6)

print(a)
print(b)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [11]:
b_numpy = np.arange(6)

print(b.__class__)
print(b_numpy.__class__)

<class 'jaxlib._jax.ArrayImpl'>
<class 'numpy.ndarray'>


JAX and numpy use **different types** to represent arrays. 

In contrast to NumPy, JAX can **execute the same code on different backends** – CPU, GPU, and TPU. 

Similar to PyTorch, we can check the device of an array by calling `.devices()`:

In [15]:
b.devices()

{CpuDevice(id=0)}

In [ ]:
b_cpu = jax.device_get(b)
print(b_cpu.__class__)

<class 'numpy.ndarray'>


In [18]:
b_gpu = jax.device_put(b_cpu)
print(f"Device put: {b_gpu.__class__} on {b_gpu.devices()}")

Device put: <class 'jaxlib._jax.ArrayImpl'> on {CpuDevice(id=0)}
